In [1]:
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SPLITS_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits"
MODELS_T_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-models-cic-scenario-t"
OUTPUT_DIR = "/kaggle/working/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

d = os.path.join(SPLITS_ROOT, "cic_ids2017", "scenario_T")
cic_T = {
    "train": pd.read_parquet(os.path.join(d, "train.parquet")),
    "id":    pd.read_parquet(os.path.join(d, "id.parquet")),
    "id2":   pd.read_parquet(os.path.join(d, "id2.parquet")),
    "ood":   pd.read_parquet(os.path.join(d, "ood.parquet")),
}
with open(os.path.join(SPLITS_ROOT, "cic_ids2017", "config.json")) as f:
    cic_config = json.load(f)
CIC_FEATURES = cic_config["model_features"]
print(f"scenario_T: Train={cic_T['train'].shape}, ID={cic_T['id'].shape}, ID2={cic_T['id2'].shape}, OOD={cic_T['ood'].shape}")

def make_fit_val_split(train_df, val_frac=0.10, seed=SEED):
    shuffled = train_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n_val = int(len(shuffled) * val_frac)
    return shuffled.iloc[n_val:].copy(), shuffled.iloc[:n_val].copy()

fit_T, val_T = make_fit_val_split(cic_T["train"])
print(f"Rebuilt fit/val: fit={len(fit_T)}, val={len(val_T)}")

class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        enc_layers = []; prev = input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        enc_layers += [nn.Linear(prev, latent_dim)]
        self.encoder = nn.Sequential(*enc_layers)
        dec_layers = []; prev = latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        dec_layers += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*dec_layers)
    def forward(self, x):
        return self.decoder(self.encoder(x))

class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        enc_layers = []; prev = input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        self.encoder_body = nn.Sequential(*enc_layers)
        self.fc_mu = nn.Linear(prev, latent_dim)
        self.fc_logvar = nn.Linear(prev, latent_dim)
        dec_layers = []; prev = latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        self.decoder_body = nn.Sequential(*dec_layers)
        self.fc_out_mu = nn.Linear(prev, input_dim)
        self.fc_out_logvar = nn.Linear(prev, input_dim)
    def encode(self, x):
        h = self.encoder_body(x)
        return self.fc_mu(h), self.fc_logvar(h)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std
    def decode(self, z):
        h = self.decoder_body(z)
        return self.fc_out_mu(h), torch.clamp(self.fc_out_logvar(h), min=-10, max=10)
    def forward(self, x):
        mu_z, logvar_z = self.encode(x)
        z = self.reparameterize(mu_z, logvar_z)
        out_mu, out_logvar = self.decode(z)
        return out_mu, out_logvar, mu_z, logvar_z

def vae_elbo_loss(x, out_mu, out_logvar, mu_z, logvar_z):
    recon_nll = 0.5 * (out_logvar + ((x - out_mu) ** 2) / torch.exp(out_logvar) + np.log(2 * np.pi))
    recon_nll = recon_nll.sum(dim=1).mean()
    kl = -0.5 * torch.sum(1 + logvar_z - mu_z.pow(2) - logvar_z.exp(), dim=1).mean()
    return recon_nll + kl, recon_nll, kl

def train_vae(fit_df, val_df, feature_cols, arch_config, train_config, device, verbose=True, log_every=5):
    input_dim = len(feature_cols)
    model = VAE(input_dim, arch_config["hidden_dims"], arch_config["latent_dim"]).to(device)
    X_fit = torch.tensor(fit_df[feature_cols].values, dtype=torch.float32)
    X_val = torch.tensor(val_df[feature_cols].values, dtype=torch.float32).to(device)
    fit_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_fit), batch_size=train_config["batch_size"], shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=train_config["learning_rate"])
    best_val_elbo = float("inf"); best_state = None; epochs_no_improve = 0
    history = {"train_elbo": [], "val_elbo": [], "val_recon_nll": [], "val_kl": []}
    for epoch in range(1, train_config["max_epochs"] + 1):
        model.train(); epoch_losses = []
        for (batch,) in fit_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out_mu, out_logvar, mu_z, logvar_z = model(batch)
            loss, _, _ = vae_elbo_loss(batch, out_mu, out_logvar, mu_z, logvar_z)
            loss.backward(); optimizer.step()
            epoch_losses.append(loss.item())
        train_elbo = float(np.mean(epoch_losses))
        model.eval()
        with torch.no_grad():
            out_mu, out_logvar, mu_z, logvar_z = model(X_val)
            val_elbo, val_recon_nll, val_kl = vae_elbo_loss(X_val, out_mu, out_logvar, mu_z, logvar_z)
            val_elbo, val_recon_nll, val_kl = val_elbo.item(), val_recon_nll.item(), val_kl.item()
        history["train_elbo"].append(train_elbo); history["val_elbo"].append(val_elbo)
        history["val_recon_nll"].append(val_recon_nll); history["val_kl"].append(val_kl)
        if verbose and (epoch == 1 or epoch % log_every == 0):
            print(f"  Epoch {epoch:3d}  train_elbo={train_elbo:.4f}  val_elbo={val_elbo:.4f}  (recon_nll={val_recon_nll:.4f}, kl={val_kl:.4f})")
        if val_elbo < best_val_elbo - train_config["min_delta"]:
            best_val_elbo = val_elbo; best_state = {k: v.clone() for k, v in model.state_dict().items()}; epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= train_config["patience"]:
                if verbose: print(f"  Early stopping at epoch {epoch} (no improvement for {train_config['patience']} epochs)")
                break
    model.load_state_dict(best_state)
    return model, history, best_val_elbo, epoch

print("\nReload complete: data, fit/val, and classes ready.")

Device: cuda
scenario_T: Train=(910016, 82), ID=(296457, 82), ID2=(296457, 82), OOD=(1017868, 82)
Rebuilt fit/val: fit=819015, val=91001

Reload complete: data, fit/val, and classes ready.


In [2]:
ARCH_CONFIG_CIC = {"hidden_dims": [48, 32], "latent_dim": 16}
ae_cic_T = Autoencoder(len(CIC_FEATURES), ARCH_CONFIG_CIC["hidden_dims"], ARCH_CONFIG_CIC["latent_dim"]).to(DEVICE)
ae_cic_T.load_state_dict(torch.load(os.path.join(MODELS_T_ROOT, "ae_cic_scenario_T", "model.pt"), map_location=DEVICE))
ae_cic_T.eval()

saved_id_scores = pd.read_parquet(os.path.join(MODELS_T_ROOT, "ae_cic_scenario_T", "scores_id.parquet"))

X_id = torch.tensor(cic_T["id"][CIC_FEATURES].values, dtype=torch.float32).to(DEVICE)
with torch.no_grad():
    recon = ae_cic_T(X_id)
    fresh_errors = ((recon - X_id) ** 2).mean(dim=1).cpu().numpy()

max_diff = np.abs(fresh_errors - saved_id_scores["reconstruction_error"].values).max()
print(f"Max difference between reloaded and saved ID scores: {max_diff:.10f}")
print("Match:", max_diff < 1e-5)

Max difference between reloaded and saved ID scores: 0.0000000000
Match: True


In [3]:
ARCH_CONFIG_CIC_VAE = {
    "hidden_dims": [48, 32],
    "latent_dim": 16,
    "architecture_rationale": "Mirrors AE-CIC exactly (67->48->32->16->32->48->67) for a fair comparison; only VAE-specific heads (mu/logvar) differ",
}

TRAIN_CONFIG_CIC = {
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "batch_size": 256,
    "max_epochs": 200,
    "patience": 10,
    "min_delta": 1e-5,
    "early_stopping_criterion": "validation ELBO, no improvement > min_delta for `patience` consecutive epochs",
    "seed": SEED,
    "loss": "ELBO (reconstruction NLL + KL)",
}
print(json.dumps(ARCH_CONFIG_CIC_VAE, indent=2))

print("\nTraining VAE — CIC-IDS2017 scenario_T")
vae_cic_T, history_vae_cic_T, best_val_elbo_cic_T, stopped_epoch_vae_cic_T = train_vae(
    fit_df=fit_T,
    val_df=val_T,
    feature_cols=CIC_FEATURES,
    arch_config=ARCH_CONFIG_CIC_VAE,
    train_config=TRAIN_CONFIG_CIC,
    device=DEVICE,
    log_every=10,
)
print(f"\nFinished at epoch {stopped_epoch_vae_cic_T}, best val_elbo={best_val_elbo_cic_T:.4f}")

{
  "hidden_dims": [
    48,
    32
  ],
  "latent_dim": 16,
  "architecture_rationale": "Mirrors AE-CIC exactly (67->48->32->16->32->48->67) for a fair comparison; only VAE-specific heads (mu/logvar) differ"
}

Training VAE — CIC-IDS2017 scenario_T
  Epoch   1  train_elbo=25204.4071  val_elbo=-104.5034  (recon_nll=-130.2691, kl=25.7657)
  Epoch  10  train_elbo=691.4449  val_elbo=-117.8149  (recon_nll=-152.1055, kl=34.2906)
  Early stopping at epoch 16 (no improvement for 10 epochs)

Finished at epoch 16, best val_elbo=-173.1427


In [4]:
def per_feature_variance_diagnostic(model, df, feature_cols, device):
    X = torch.tensor(df[feature_cols].values, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        mu_z, logvar_z = model.encode(X)
        _, out_logvar = model.decode(mu_z)  # use mean z, no sampling noise
    out_logvar = out_logvar.cpu().numpy()

    per_feature_mean = out_logvar.mean(axis=0)
    per_feature_min = out_logvar.min(axis=0)
    pct_clamped_per_feature = (out_logvar <= -9.9).mean(axis=0) * 100

    result = pd.DataFrame({
        "feature": feature_cols,
        "mean_logvar": per_feature_mean,
        "min_logvar": per_feature_min,
        "pct_at_clamp": pct_clamped_per_feature,
    }).sort_values("mean_logvar")

    return result

diag = per_feature_variance_diagnostic(vae_cic_T, val_T, CIC_FEATURES, DEVICE)
print("10 most extreme (lowest mean_logvar) features:")
print(diag.head(10).to_string(index=False))
print(f"\nFeatures with >10% of values at clamp: {(diag['pct_at_clamp'] > 10).sum()}")
print(diag[diag["pct_at_clamp"] > 10].to_string(index=False))

print("\nFull training history:")
for i, (te, ve, nll, kl) in enumerate(zip(
    history_vae_cic_T["train_elbo"], history_vae_cic_T["val_elbo"],
    history_vae_cic_T["val_recon_nll"], history_vae_cic_T["val_kl"]
), 1):
    print(f"  Epoch {i:3d}  train_elbo={te:12.4f}  val_elbo={ve:10.4f}  recon_nll={nll:10.4f}  kl={kl:8.4f}")

10 most extreme (lowest mean_logvar) features:
                    feature  mean_logvar  min_logvar  pct_at_clamp
             RST Flag Count    -9.974783       -10.0     99.641762
             ECE Flag Count    -9.966574       -10.0     99.425281
          Bwd Header Length    -9.959118       -10.0     98.484632
       min_seg_size_forward    -9.908706       -10.0     96.815420
          Subflow Bwd Bytes    -9.865977       -10.0     95.153899
Total Length of Bwd Packets    -9.862424       -10.0     95.116537
           act_data_pkt_fwd    -9.826637       -10.0     91.786903
        Subflow Bwd Packets    -9.817539       -10.0     92.609971
     Total Backward Packets    -9.816941       -10.0     92.625356
          Total Fwd Packets    -9.814133       -10.0     91.662729

Features with >10% of values at clamp: 65
                    feature  mean_logvar  min_logvar  pct_at_clamp
             RST Flag Count    -9.974783       -10.0     99.641762
             ECE Flag Count    -9.96657

In [5]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        enc_layers = []; prev = input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        self.encoder_body = nn.Sequential(*enc_layers)
        self.fc_mu = nn.Linear(prev, latent_dim)
        self.fc_logvar = nn.Linear(prev, latent_dim)

        dec_layers = []; prev = latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        dec_layers += [nn.Linear(prev, input_dim)]
        self.decoder_body = nn.Sequential(*dec_layers)  # outputs reconstruction mean directly now

        # Single global scalar log-variance, shared across all features and all rows —
        # replaces the per-feature learned variance that caused the collapse
        self.global_logvar = nn.Parameter(torch.zeros(1))

    def encode(self, x):
        h = self.encoder_body(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        return self.decoder_body(z)

    def forward(self, x):
        mu_z, logvar_z = self.encode(x)
        z = self.reparameterize(mu_z, logvar_z)
        out_mu = self.decode(z)
        return out_mu, mu_z, logvar_z


def vae_elbo_loss(x, out_mu, mu_z, logvar_z, global_logvar):
    logvar = torch.clamp(global_logvar, min=-10, max=10)
    recon_nll = 0.5 * (logvar + ((x - out_mu) ** 2) / torch.exp(logvar) + np.log(2 * np.pi))
    recon_nll = recon_nll.sum(dim=1).mean()
    kl = -0.5 * torch.sum(1 + logvar_z - mu_z.pow(2) - logvar_z.exp(), dim=1).mean()
    return recon_nll + kl, recon_nll, kl


def train_vae(fit_df, val_df, feature_cols, arch_config, train_config, device, verbose=True, log_every=5):
    input_dim = len(feature_cols)
    model = VAE(input_dim, arch_config["hidden_dims"], arch_config["latent_dim"]).to(device)
    X_fit = torch.tensor(fit_df[feature_cols].values, dtype=torch.float32)
    X_val = torch.tensor(val_df[feature_cols].values, dtype=torch.float32).to(device)
    fit_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_fit), batch_size=train_config["batch_size"], shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=train_config["learning_rate"])

    best_val_elbo = float("inf"); best_state = None; epochs_no_improve = 0
    history = {"train_elbo": [], "val_elbo": [], "val_recon_nll": [], "val_kl": [], "global_logvar": []}

    for epoch in range(1, train_config["max_epochs"] + 1):
        model.train(); epoch_losses = []
        for (batch,) in fit_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out_mu, mu_z, logvar_z = model(batch)
            loss, _, _ = vae_elbo_loss(batch, out_mu, mu_z, logvar_z, model.global_logvar)
            loss.backward(); optimizer.step()
            epoch_losses.append(loss.item())
        train_elbo = float(np.mean(epoch_losses))

        model.eval()
        with torch.no_grad():
            out_mu, mu_z, logvar_z = model(X_val)
            val_elbo, val_recon_nll, val_kl = vae_elbo_loss(X_val, out_mu, mu_z, logvar_z, model.global_logvar)
            val_elbo, val_recon_nll, val_kl = val_elbo.item(), val_recon_nll.item(), val_kl.item()

        history["train_elbo"].append(train_elbo); history["val_elbo"].append(val_elbo)
        history["val_recon_nll"].append(val_recon_nll); history["val_kl"].append(val_kl)
        history["global_logvar"].append(model.global_logvar.item())

        if verbose and (epoch == 1 or epoch % log_every == 0):
            print(f"  Epoch {epoch:3d}  train_elbo={train_elbo:10.4f}  val_elbo={val_elbo:10.4f}  "
                  f"(recon_nll={val_recon_nll:10.4f}, kl={val_kl:8.4f}, global_logvar={model.global_logvar.item():.4f})")

        if val_elbo < best_val_elbo - train_config["min_delta"]:
            best_val_elbo = val_elbo; best_state = {k: v.clone() for k, v in model.state_dict().items()}; epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= train_config["patience"]:
                if verbose: print(f"  Early stopping at epoch {epoch} (no improvement for {train_config['patience']} epochs)")
                break

    model.load_state_dict(best_state)
    return model, history, best_val_elbo, epoch

print("VAE redefined with global scalar variance.")

VAE redefined with global scalar variance.


In [6]:
ARCH_CONFIG_CIC_VAE = {
    "hidden_dims": [48, 32],
    "latent_dim": 16,
    "architecture_rationale": "Mirrors AE-CIC exactly (67->48->32->16->32->48->67); global scalar output variance instead of per-feature, to prevent variance collapse on near-constant flag/header columns",
    "output_variance": "global scalar, shared across all features and rows",
}
print(json.dumps(ARCH_CONFIG_CIC_VAE, indent=2))

print("\nTraining VAE — CIC-IDS2017 scenario_T (global variance)")
vae_cic_T, history_vae_cic_T, best_val_elbo_cic_T, stopped_epoch_vae_cic_T = train_vae(
    fit_df=fit_T,
    val_df=val_T,
    feature_cols=CIC_FEATURES,
    arch_config=ARCH_CONFIG_CIC_VAE,
    train_config=TRAIN_CONFIG_CIC,
    device=DEVICE,
    log_every=10,
)
print(f"\nFinished at epoch {stopped_epoch_vae_cic_T}, best val_elbo={best_val_elbo_cic_T:.4f}")

{
  "hidden_dims": [
    48,
    32
  ],
  "latent_dim": 16,
  "architecture_rationale": "Mirrors AE-CIC exactly (67->48->32->16->32->48->67); global scalar output variance instead of per-feature, to prevent variance collapse on near-constant flag/header columns",
  "output_variance": "global scalar, shared across all features and rows"
}

Training VAE — CIC-IDS2017 scenario_T (global variance)
  Epoch   1  train_elbo=   73.1791  val_elbo=   54.2745  (recon_nll=   49.0998, kl=  5.1748, global_logvar=-0.7211)
  Epoch  10  train_elbo=41158.4335  val_elbo=   35.9730  (recon_nll=   29.6581, kl=  6.3149, global_logvar=-1.2973)
  Epoch  20  train_elbo=   30.8271  val_elbo=   23.4198  (recon_nll=   16.2955, kl=  7.1243, global_logvar=-1.7187)
  Epoch  30  train_elbo=  485.0784  val_elbo=   17.0950  (recon_nll=    8.8030, kl=  8.2920, global_logvar=-2.0234)
  Epoch  40  train_elbo=   40.6366  val_elbo=   19.5710  (recon_nll=    9.7862, kl=  9.7849, global_logvar=-2.1484)
  Early stopping at ep

In [7]:
def compute_vae_nll(model, df, feature_cols, device, n_samples=10):
    X = torch.tensor(df[feature_cols].values, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        mu_z, logvar_z = model.encode(X)
        logvar = torch.clamp(model.global_logvar, min=-10, max=10)
        nlls = []
        for _ in range(n_samples):
            z = model.reparameterize(mu_z, logvar_z)
            out_mu = model.decode(z)
            nll = 0.5 * (logvar + ((X - out_mu) ** 2) / torch.exp(logvar) + np.log(2 * np.pi))
            nlls.append(nll.sum(dim=1))
        avg_nll = torch.stack(nlls).mean(dim=0).cpu().numpy()
    return avg_nll

def score_split_vae(model, df, feature_cols, device, split_name, n_samples=10):
    scores = compute_vae_nll(model, df, feature_cols, device, n_samples)
    out = pd.DataFrame({"row_id": np.arange(len(df)), "Class": df["Class"].values, "vae_nll": scores})
    benign = out[out.Class == 0].vae_nll
    anomaly = out[out.Class == 1].vae_nll
    print(f"{split_name}: benign mean={benign.mean():.4f} median={benign.median():.4f} | "
          f"anomaly mean={anomaly.mean():.4f} median={anomaly.median():.4f} "
          f"(n_benign={len(benign)}, n_anomaly={len(anomaly)})")
    return out

print("=== VAE — scenario_T NLL by split/class (mean and median) ===")
vae_cic_T_scores = {
    "id":  score_split_vae(vae_cic_T, cic_T["id"],  CIC_FEATURES, DEVICE, "ID"),
    "id2": score_split_vae(vae_cic_T, cic_T["id2"], CIC_FEATURES, DEVICE, "ID2"),
    "ood": score_split_vae(vae_cic_T, cic_T["ood"], CIC_FEATURES, DEVICE, "OOD-temporal"),
}

=== VAE — scenario_T NLL by split/class (mean and median) ===
ID: benign mean=8.8860 median=-0.5306 | anomaly mean=826.2101 median=495.5887 (n_benign=195004, n_anomaly=101453)
ID2: benign mean=162.7106 median=-0.5161 | anomaly mean=825.0623 median=494.4178 (n_benign=195004, n_anomaly=101453)
OOD-temporal: benign mean=inf median=0.4038 | anomaly mean=5927.9321 median=21.0965 (n_benign=795033, n_anomaly=222835)


In [8]:
def compute_vae_nll_debug(model, df, feature_cols, device, n_samples=10):
    X = torch.tensor(df[feature_cols].values, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        mu_z, logvar_z = model.encode(X)
        logvar = torch.clamp(model.global_logvar, min=-10, max=10)
        z = mu_z  # deterministic, no sampling, for inspecting the worst offenders
        out_mu = model.decode(z)
        sq_error = (X - out_mu) ** 2
        nll_per_row = (0.5 * (logvar + sq_error / torch.exp(logvar) + np.log(2 * np.pi))).sum(dim=1)
    return nll_per_row.cpu().numpy(), sq_error.cpu().numpy()

nll_ood, sq_error_ood = compute_vae_nll_debug(vae_cic_T, cic_T["ood"], CIC_FEATURES, DEVICE)

n_inf = np.isinf(nll_ood).sum()
n_nan = np.isnan(nll_ood).sum()
print(f"Inf values: {n_inf}, NaN values: {n_nan}, out of {len(nll_ood)} rows")

finite_mask = np.isfinite(nll_ood)
print(f"\nOf finite values — max: {nll_ood[finite_mask].max():.2f}, p99.9: {np.percentile(nll_ood[finite_mask], 99.9):.2f}")

if n_inf > 0 or n_nan > 0:
    bad_idx = np.where(~finite_mask)[0][:3]
    for idx in bad_idx:
        worst_features = np.argsort(sq_error_ood[idx])[-5:][::-1]
        print(f"\nRow {idx} (Class={cic_T['ood'].iloc[idx]['Class']}) — top 5 squared errors:")
        for fi in worst_features:
            print(f"    {CIC_FEATURES[fi]}: raw_value={cic_T['ood'].iloc[idx][CIC_FEATURES[fi]]:.2e}, "
                  f"scaled_value={cic_T['ood'][CIC_FEATURES[fi]].values[idx]:.2e}, sq_error={sq_error_ood[idx][fi]:.2e}")

Inf values: 0, NaN values: 0, out of 1017868 rows

Of finite values — max: 1563218.00, p99.9: 6763.71


In [9]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        enc_layers = []; prev = input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        self.encoder_body = nn.Sequential(*enc_layers)
        self.fc_mu = nn.Linear(prev, latent_dim)
        self.fc_logvar = nn.Linear(prev, latent_dim)

        dec_layers = []; prev = latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        dec_layers += [nn.Linear(prev, input_dim)]
        self.decoder_body = nn.Sequential(*dec_layers)

        self.global_logvar = nn.Parameter(torch.zeros(1))

    def encode(self, x):
        h = self.encoder_body(x)
        mu = self.fc_mu(h)
        logvar = torch.clamp(self.fc_logvar(h), min=-10, max=10)  # NEW: prevents pathological z samples
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        return self.decoder_body(z)

    def forward(self, x):
        mu_z, logvar_z = self.encode(x)
        z = self.reparameterize(mu_z, logvar_z)
        out_mu = self.decode(z)
        return out_mu, mu_z, logvar_z

print("VAE redefined with logvar_z clamping. Retraining scenario_T...")
vae_cic_T, history_vae_cic_T, best_val_elbo_cic_T, stopped_epoch_vae_cic_T = train_vae(
    fit_df=fit_T,
    val_df=val_T,
    feature_cols=CIC_FEATURES,
    arch_config=ARCH_CONFIG_CIC_VAE,
    train_config=TRAIN_CONFIG_CIC,
    device=DEVICE,
    log_every=10,
)
print(f"\nFinished at epoch {stopped_epoch_vae_cic_T}, best val_elbo={best_val_elbo_cic_T:.4f}")

# Check: did train_elbo spikes disappear?
train_elbos = history_vae_cic_T["train_elbo"]
print(f"\nMax train_elbo across all epochs: {max(train_elbos):.2f} (should be a normal small number now, not thousands/millions)")

VAE redefined with logvar_z clamping. Retraining scenario_T...
  Epoch   1  train_elbo=   72.3121  val_elbo=   52.9731  (recon_nll=   47.6206, kl=  5.3525, global_logvar=-0.7257)
  Epoch  10  train_elbo=   32.4795  val_elbo=   11.1396  (recon_nll=    2.0298, kl=  9.1099, global_logvar=-2.2597)
  Epoch  20  train_elbo=   24.6915  val_elbo=   -2.8946  (recon_nll=  -13.5499, kl= 10.6553, global_logvar=-2.7387)
  Epoch  30  train_elbo=    8.8005  val_elbo=   -7.5652  (recon_nll=  -18.7604, kl= 11.1952, global_logvar=-2.9474)
  Epoch  40  train_elbo=    9.7933  val_elbo=  -10.7958  (recon_nll=  -22.1513, kl= 11.3555, global_logvar=-3.1355)
  Epoch  50  train_elbo=   85.1265  val_elbo=   -2.4218  (recon_nll=  -15.9395, kl= 13.5177, global_logvar=-2.9658)
  Early stopping at epoch 52 (no improvement for 10 epochs)

Finished at epoch 52, best val_elbo=-15.0885

Max train_elbo across all epochs: 85.13 (should be a normal small number now, not thousands/millions)


In [10]:
print("=== VAE — scenario_T NLL by split/class (fixed model) ===")
vae_cic_T_scores = {
    "id":  score_split_vae(vae_cic_T, cic_T["id"],  CIC_FEATURES, DEVICE, "ID"),
    "id2": score_split_vae(vae_cic_T, cic_T["id2"], CIC_FEATURES, DEVICE, "ID2"),
    "ood": score_split_vae(vae_cic_T, cic_T["ood"], CIC_FEATURES, DEVICE, "OOD-temporal"),
}

# Explicit check: are all values finite now?
for name, df_scores in vae_cic_T_scores.items():
    n_inf = np.isinf(df_scores["vae_nll"]).sum()
    n_nan = np.isnan(df_scores["vae_nll"]).sum()
    print(f"{name}: inf={n_inf}, nan={n_nan}")

=== VAE — scenario_T NLL by split/class (fixed model) ===
ID: benign mean=-25.3319 median=-39.6432 | anomaly mean=1971.3463 median=1395.1379 (n_benign=195004, n_anomaly=101453)
ID2: benign mean=-15.9063 median=-39.6314 | anomaly mean=1970.9429 median=1391.6498 (n_benign=195004, n_anomaly=101453)
OOD-temporal: benign mean=14.1566 median=-39.5084 | anomaly mean=2093.2041 median=-16.0789 (n_benign=795033, n_anomaly=222835)
id: inf=0, nan=0
id2: inf=0, nan=0
ood: inf=0, nan=0


In [11]:
def score_percentiles(model, df, feature_cols, device, class_val, split_name, n_samples=10):
    subset = df[df["Class"] == class_val]
    scores = compute_vae_nll(model, subset, feature_cols, device, n_samples)
    pcts = [1, 5, 25, 50, 75, 95, 99]
    vals = np.percentile(scores, pcts)
    print(f"{split_name} (Class={class_val}, n={len(scores)}):")
    for p, v in zip(pcts, vals):
        print(f"    p{p:2d} = {v:10.2f}")
    return scores

print("=== OOD-temporal anomaly score distribution ===")
ood_anomaly_scores = score_percentiles(vae_cic_T, cic_T["ood"], CIC_FEATURES, DEVICE, 1, "OOD-temporal anomalies")

print("\n=== OOD-temporal benign score distribution (for comparison) ===")
ood_benign_scores = score_percentiles(vae_cic_T, cic_T["ood"], CIC_FEATURES, DEVICE, 0, "OOD-temporal benign")

# Which attack categories make up OOD-temporal's anomaly class, and do they score differently?
ood_with_scores = cic_T["ood"][cic_T["ood"]["Class"] == 1].copy()
ood_with_scores["vae_nll"] = ood_anomaly_scores
print("\n=== Median NLL by attack category within OOD-temporal ===")
print(ood_with_scores.groupby("Label")["vae_nll"].agg(["median", "count"]).sort_values("median"))

=== OOD-temporal anomaly score distribution ===
OOD-temporal anomalies (Class=1, n=222835):
    p 1 =     -41.08
    p 5 =     -38.25
    p25 =     -29.97
    p50 =     -16.12
    p75 =    1005.28
    p95 =   11331.50
    p99 =   28904.73

=== OOD-temporal benign score distribution (for comparison) ===
OOD-temporal benign (Class=0, n=795033):
    p 1 =     -43.01
    p 5 =     -42.73
    p25 =     -41.92
    p50 =     -39.51
    p75 =     -30.40
    p95 =     114.69
    p99 =     885.24

=== Median NLL by attack category within OOD-temporal ===
                                median   count
Label                                         
Web Attack - XSS            -38.549553     652
Web Attack - Brute Force    -38.296139    1470
Bot                         -35.557175    1948
Web Attack - Sql Injection  -29.164759      21
PortScan                    -27.981113   90694
Infiltration                204.000717      36
DDoS                        700.309937  128014


In [13]:
def compute_reconstruction_error(model, df, feature_cols, device):
    X = torch.tensor(df[feature_cols].values, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        recon = model(X)
        errors = ((recon - X) ** 2).mean(dim=1).cpu().numpy()
    return errors

ae_ood_errors = compute_reconstruction_error(ae_cic_T, cic_T["ood"], CIC_FEATURES, DEVICE)
ood_with_ae_scores = cic_T["ood"][cic_T["ood"]["Class"] == 1].copy()
ood_with_ae_scores["ae_error"] = ae_ood_errors[cic_T["ood"]["Class"].values == 1]

print("=== AE-T: Median reconstruction error by attack category within OOD-temporal ===")
print(ood_with_ae_scores.groupby("Label")["ae_error"].agg(["median", "count"]).sort_values("median"))

=== AE-T: Median reconstruction error by attack category within OOD-temporal ===
                              median   count
Label                                       
Web Attack - XSS            0.001053     652
Web Attack - Brute Force    0.001145    1470
Web Attack - Sql Injection  0.005521      21
Bot                         0.005527    1948
PortScan                    0.020405   90694
Infiltration                0.094438      36
DDoS                        0.405968  128014


In [14]:
VAE_T_DIR = os.path.join(OUTPUT_DIR, "vae_cic_scenario_T")
os.makedirs(VAE_T_DIR, exist_ok=True)

torch.save(vae_cic_T.state_dict(), os.path.join(VAE_T_DIR, "model.pt"))

vae_cic_T_config = {
    "model": "VAE",
    "dataset": "cic_ids2017",
    "scenario": "scenario_T (temporal)",
    "architecture": ARCH_CONFIG_CIC_VAE,
    "training": TRAIN_CONFIG_CIC,
    "stopped_epoch": stopped_epoch_vae_cic_T,
    "best_val_elbo": best_val_elbo_cic_T,
    "scoring_method": "negative log-likelihood, Monte Carlo averaged over 10 latent samples, global scalar output variance",
    "fit_size": len(fit_T),
    "val_size": len(val_T),
    "debugging_notes": [
        "Initial per-feature output variance caused numerical instability (train_elbo spiking to 1e10+) due to 65/67 features being near-constant and hitting the variance floor. Fixed by switching to a single global scalar output variance.",
        "After that fix, OOD-temporal benign mean NLL was still +inf due to unclamped logvar_z allowing pathological latent samples on outlier rows. Fixed by clamping logvar_z to [-10, 10] in encode(), same as out_logvar already was.",
    ],
    "outlier_note": "Mean NLL is unreliable on this dataset due to extreme-tail rows; use median for all ID/ID2/OOD comparisons in NB6/NB7.",
    "per_category_finding": {
        "description": "OOD-temporal anomaly detection is NOT uniform across attack categories. Web Attacks (XSS/Brute Force/SQLi), Bot, and PortScan score near-indistinguishable from benign; DDoS and Infiltration remain clearly detectable. This pattern is consistent across both AE and VAE scoring mechanisms.",
        "vae_median_by_category": {
            "Web Attack - XSS": -38.55, "Web Attack - Brute Force": -38.30, "Bot": -35.56,
            "Web Attack - Sql Injection": -29.16, "PortScan": -27.98,
            "Infiltration": 204.00, "DDoS": 700.31,
        },
        "ae_median_by_category": {
            "Web Attack - XSS": 0.00105, "Web Attack - Brute Force": 0.00115,
            "Web Attack - Sql Injection": 0.00552, "Bot": 0.00553, "PortScan": 0.02041,
            "Infiltration": 0.09444, "DDoS": 0.40597,
        },
        "implication_for_NB6_NB7": "Baseline-control and significance-testing notebooks should compute per-attack-category breakdowns for CIC-IDS2017, not just an aggregate benign-vs-anomaly comparison.",
    },
}
with open(os.path.join(VAE_T_DIR, "config.json"), "w") as f:
    json.dump(vae_cic_T_config, f, indent=2)

for split_name, scores_df in vae_cic_T_scores.items():
    scores_df.to_parquet(os.path.join(VAE_T_DIR, f"scores_{split_name}.parquet"), index=False)

print("Saved to:", VAE_T_DIR)
print(os.listdir(VAE_T_DIR))

Saved to: /kaggle/working/models/vae_cic_scenario_T
['scores_id2.parquet', 'scores_ood.parquet', 'config.json', 'model.pt', 'scores_id.parquet']


In [15]:
CC_ROOT = os.path.join(SPLITS_ROOT, "credit_card")
cc_train = pd.read_parquet(os.path.join(CC_ROOT, "train.parquet"))
cc_id    = pd.read_parquet(os.path.join(CC_ROOT, "id.parquet"))
cc_id2   = pd.read_parquet(os.path.join(CC_ROOT, "id2.parquet"))
cc_ood   = pd.read_parquet(os.path.join(CC_ROOT, "ood.parquet"))
with open(os.path.join(CC_ROOT, "config.json")) as f:
    cc_config = json.load(f)
CC_FEATURES = cc_config["model_features"]
print(f"Credit card: Train={cc_train.shape}, ID={cc_id.shape}, ID2={cc_id2.shape}, OOD={cc_ood.shape}")

fit_cc, val_cc = make_fit_val_split(cc_train)
print(f"fit={len(fit_cc)}, val={len(val_cc)}")

Credit card: Train=(100774, 31), ID=(21731, 31), ID2=(21731, 31), OOD=(139490, 31)
fit=90697, val=10077


In [17]:
TRAIN_CONFIG = {
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "batch_size": 256,
    "max_epochs": 300,
    "patience": 10,
    "min_delta": 1e-5,
    "early_stopping_criterion": "validation ELBO, no improvement > min_delta for `patience` consecutive epochs",
    "seed": SEED,
    "loss": "ELBO (reconstruction NLL + KL)",
}
print(json.dumps(TRAIN_CONFIG, indent=2))

print("\nRetraining VAE — Credit Card (fixed architecture)")
vae_cc, history_vae_cc, best_val_elbo_cc, stopped_epoch_vae_cc = train_vae(
    fit_df=fit_cc, val_df=val_cc, feature_cols=CC_FEATURES,
    arch_config=ARCH_CONFIG_CC_VAE, train_config=TRAIN_CONFIG, device=DEVICE, log_every=10,
)
print(f"\nFinished at epoch {stopped_epoch_vae_cc}, best val_elbo={best_val_elbo_cc:.4f}")
print(f"Max train_elbo across all epochs: {max(history_vae_cc['train_elbo']):.2f} (sanity check — should be small/normal)")

print("\n=== VAE — Credit Card NLL by split/class (fixed model) ===")
vae_cc_scores = {
    "id":  score_split_vae(vae_cc, cc_id,  CC_FEATURES, DEVICE, "ID"),
    "id2": score_split_vae(vae_cc, cc_id2, CC_FEATURES, DEVICE, "ID2"),
    "ood": score_split_vae(vae_cc, cc_ood, CC_FEATURES, DEVICE, "OOD"),
}
for name, df_scores in vae_cc_scores.items():
    print(f"{name}: inf={np.isinf(df_scores['vae_nll']).sum()}, nan={np.isnan(df_scores['vae_nll']).sum()}")

{
  "optimizer": "Adam",
  "learning_rate": 0.001,
  "batch_size": 256,
  "max_epochs": 300,
  "patience": 10,
  "min_delta": 1e-05,
  "early_stopping_criterion": "validation ELBO, no improvement > min_delta for `patience` consecutive epochs",
  "seed": 42,
  "loss": "ELBO (reconstruction NLL + KL)"
}

Retraining VAE — Credit Card (fixed architecture)
  Epoch   1  train_elbo=   41.1711  val_elbo=   41.5403  (recon_nll=   41.5260, kl=  0.0143, global_logvar=-0.0000)
  Epoch  10  train_elbo=   40.3650  val_elbo=   40.6603  (recon_nll=   40.1387, kl=  0.5215, global_logvar=-0.0841)
  Epoch  20  train_elbo=   39.8496  val_elbo=   40.1486  (recon_nll=   39.3408, kl=  0.8077, global_logvar=-0.1414)
  Epoch  30  train_elbo=   39.0524  val_elbo=   39.2257  (recon_nll=   37.9936, kl=  1.2321, global_logvar=-0.2205)
  Epoch  40  train_elbo=   38.8040  val_elbo=   38.9849  (recon_nll=   37.6584, kl=  1.3265, global_logvar=-0.2537)
  Epoch  50  train_elbo=   38.7287  val_elbo=   38.9067  (recon_nl

In [18]:
VAE_DIR = os.path.join(OUTPUT_DIR, "vae_credit_card")
os.makedirs(VAE_DIR, exist_ok=True)

torch.save(vae_cc.state_dict(), os.path.join(VAE_DIR, "model.pt"))

vae_cc_config = {
    "model": "VAE",
    "dataset": "credit_card",
    "architecture": ARCH_CONFIG_CC_VAE,
    "training": TRAIN_CONFIG,
    "stopped_epoch": stopped_epoch_vae_cc,
    "best_val_elbo": best_val_elbo_cc,
    "scoring_method": "negative log-likelihood, Monte Carlo averaged over 10 latent samples, global scalar output variance",
    "fit_size": len(fit_cc),
    "val_size": len(val_cc),
    "superseded_note": "This replaces an earlier Credit Card VAE run that used per-feature output variance and unclamped logvar_z. That version is invalid — discard scores_*.parquet from the original vae_credit_card upload (session 1). The earlier run's near-collapse of OOD anomaly separation (1.18x) does not hold under this corrected architecture (~6.5x here) and should not be reported.",
}
with open(os.path.join(VAE_DIR, "config.json"), "w") as f:
    json.dump(vae_cc_config, f, indent=2)

for split_name, scores_df in vae_cc_scores.items():
    scores_df.to_parquet(os.path.join(VAE_DIR, f"scores_{split_name}.parquet"), index=False)

print("Saved to:", VAE_DIR)
print(os.listdir(VAE_DIR))

Saved to: /kaggle/working/models/vae_credit_card
['scores_id2.parquet', 'scores_ood.parquet', 'config.json', 'model.pt', 'scores_id.parquet']


In [19]:
import shutil
shutil.make_archive("/kaggle/working/models_archive_cc_vae_fixed", "zip", "/kaggle/working/models")
print("Size (MB):", os.path.getsize("/kaggle/working/models_archive_cc_vae_fixed.zip") / (1024*1024))

Size (MB): 9.153724670410156


In [20]:
d = os.path.join(SPLITS_ROOT, "cic_ids2017", "scenario_A")
cic_A = {
    "train": pd.read_parquet(os.path.join(d, "train.parquet")),
    "id":    pd.read_parquet(os.path.join(d, "id.parquet")),
    "id2":   pd.read_parquet(os.path.join(d, "id2.parquet")),
    "ood":   pd.read_parquet(os.path.join(d, "ood.parquet")),
}
print(f"scenario_A: Train={cic_A['train'].shape}, ID={cic_A['id'].shape}, ID2={cic_A['id2'].shape}, OOD={cic_A['ood'].shape}")

fit_A, val_A = make_fit_val_split(cic_A["train"])
print(f"fit={len(fit_A)}, val={len(val_A)}")

scenario_A: Train=(1190196, 82), ID=(357584, 82), ID2=(357585, 82), OOD=(615433, 82)
fit=1071177, val=119019


In [22]:
def train_autoencoder(fit_df, val_df, feature_cols, arch_config, train_config, device, verbose=True, log_every=5):
    input_dim = len(feature_cols)
    model = Autoencoder(input_dim, arch_config["hidden_dims"], arch_config["latent_dim"]).to(device)

    X_fit = torch.tensor(fit_df[feature_cols].values, dtype=torch.float32)
    X_val = torch.tensor(val_df[feature_cols].values, dtype=torch.float32).to(device)

    fit_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_fit),
        batch_size=train_config["batch_size"], shuffle=True
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=train_config["learning_rate"])
    loss_fn = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, train_config["max_epochs"] + 1):
        model.train()
        epoch_losses = []
        for (batch,) in fit_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            recon = model(batch)
            loss = loss_fn(recon, batch)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        train_loss = float(np.mean(epoch_losses))

        model.eval()
        with torch.no_grad():
            val_recon = model(X_val)
            val_loss = loss_fn(val_recon, X_val).item()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        if verbose and (epoch == 1 or epoch % log_every == 0):
            print(f"  Epoch {epoch:3d}  train_loss={train_loss:.6f}  val_loss={val_loss:.6f}")

        if val_loss < best_val_loss - train_config["min_delta"]:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= train_config["patience"]:
                if verbose:
                    print(f"  Early stopping at epoch {epoch} (no improvement for {train_config['patience']} epochs)")
                break

    model.load_state_dict(best_state)
    return model, history, best_val_loss, epoch


ARCH_CONFIG_CIC = {"hidden_dims": [48, 32], "latent_dim": 16}
TRAIN_CONFIG_CIC = {
    "optimizer": "Adam", "learning_rate": 1e-3, "batch_size": 256, "max_epochs": 200,
    "patience": 10, "min_delta": 1e-5,
    "early_stopping_criterion": "validation MSE, no improvement > min_delta for `patience` consecutive epochs",
    "seed": SEED, "loss": "MSE",
}

print("Training Autoencoder — CIC-IDS2017 scenario_A")
ae_cic_A, history_ae_cic_A, best_val_loss_ae_cic_A, stopped_epoch_ae_cic_A = train_autoencoder(
    fit_df=fit_A, val_df=val_A, feature_cols=CIC_FEATURES,
    arch_config=ARCH_CONFIG_CIC, train_config=TRAIN_CONFIG_CIC, device=DEVICE, log_every=10,
)
print(f"\nFinished at epoch {stopped_epoch_ae_cic_A}, best val_loss={best_val_loss_ae_cic_A:.6f}")

Training Autoencoder — CIC-IDS2017 scenario_A
  Epoch   1  train_loss=0.246975  val_loss=0.067045
  Epoch  10  train_loss=0.064810  val_loss=0.026374
  Epoch  20  train_loss=0.060424  val_loss=0.017613
  Epoch  30  train_loss=0.065167  val_loss=0.010165
  Epoch  40  train_loss=0.057460  val_loss=0.010047
  Epoch  50  train_loss=0.055081  val_loss=0.008967
  Epoch  60  train_loss=0.044607  val_loss=0.012438
  Early stopping at epoch 60 (no improvement for 10 epochs)

Finished at epoch 60, best val_loss=0.008967


In [24]:
def score_split(model, df, feature_cols, device, split_name):
    errors = compute_reconstruction_error(model, df, feature_cols, device)
    out = pd.DataFrame({
        "row_id": np.arange(len(df)),
        "Class": df["Class"].values,
        "reconstruction_error": errors,
    })
    print(f"{split_name}: benign mean={out[out.Class==0].reconstruction_error.mean():.4f}, "
          f"anomaly mean={out[out.Class==1].reconstruction_error.mean():.4f} "
          f"(n_benign={sum(out.Class==0)}, n_anomaly={sum(out.Class==1)})")
    return out

print("=== AE — scenario_A reconstruction error by split/class ===")
ae_cic_A_scores = {
    "id":  score_split(ae_cic_A, cic_A["id"],  CIC_FEATURES, DEVICE, "ID"),
    "id2": score_split(ae_cic_A, cic_A["id2"], CIC_FEATURES, DEVICE, "ID2"),
    "ood": score_split(ae_cic_A, cic_A["ood"], CIC_FEATURES, DEVICE, "OOD-attack"),
}

for split_name, split_df in [("ID", cic_A["id"]), ("ID2", cic_A["id2"]), ("OOD-attack", cic_A["ood"])]:
    errors = compute_reconstruction_error(ae_cic_A, split_df, CIC_FEATURES, DEVICE)
    benign_med = np.median(errors[split_df["Class"].values == 0])
    anomaly_med = np.median(errors[split_df["Class"].values == 1]) if (split_df["Class"] == 1).any() else float("nan")
    print(f"{split_name} median: benign={benign_med:.5f}, anomaly={anomaly_med:.5f}")

ood_A_errors = compute_reconstruction_error(ae_cic_A, cic_A["ood"], CIC_FEATURES, DEVICE)
ood_A_with_scores = cic_A["ood"][cic_A["ood"]["Class"] == 1].copy()
ood_A_with_scores["ae_error"] = ood_A_errors[cic_A["ood"]["Class"].values == 1]

print("\n=== AE-A: Median reconstruction error by unseen attack category (OOD-attack) ===")
print(ood_A_with_scores.groupby("Label")["ae_error"].agg(["median", "count"]).sort_values("median"))

=== AE — scenario_A reconstruction error by split/class ===
ID: benign mean=0.0215, anomaly mean=1.4742 (n_benign=255042, n_anomaly=102542)
ID2: benign mean=0.0183, anomaly mean=1.4644 (n_benign=255042, n_anomaly=102543)
OOD-attack: benign mean=0.0687, anomaly mean=0.7635 (n_benign=394777, n_anomaly=220656)
ID median: benign=0.00104, anomaly=0.40428
ID2 median: benign=0.00104, anomaly=0.40354
OOD-attack median: benign=0.00108, anomaly=0.01760

=== AE-A: Median reconstruction error by unseen attack category (OOD-attack) ===
            median   count
Label                     
Bot       0.005198    1948
PortScan  0.006745   90694
DDoS      0.339020  128014


In [25]:
ARCH_CONFIG_CIC_VAE_A = {
    "hidden_dims": [48, 32],
    "latent_dim": 16,
    "architecture_rationale": "Mirrors AE-A and scenario_T's VAE exactly; global scalar output variance + clamped logvar_z",
    "output_variance": "global scalar, shared across all features and rows",
}

print("Training VAE — CIC-IDS2017 scenario_A")
vae_cic_A, history_vae_cic_A, best_val_elbo_cic_A, stopped_epoch_vae_cic_A = train_vae(
    fit_df=fit_A, val_df=val_A, feature_cols=CIC_FEATURES,
    arch_config=ARCH_CONFIG_CIC_VAE_A, train_config=TRAIN_CONFIG_CIC, device=DEVICE, log_every=10,
)
print(f"\nFinished at epoch {stopped_epoch_vae_cic_A}, best val_elbo={best_val_elbo_cic_A:.4f}")
print(f"Max train_elbo across all epochs: {max(history_vae_cic_A['train_elbo']):.2f} (should be small/normal, given the logvar_z clamp)")

Training VAE — CIC-IDS2017 scenario_A
  Epoch   1  train_elbo=   69.4925  val_elbo=   51.7061  (recon_nll=   46.9090, kl=  4.7971, global_logvar=-0.7520)
  Epoch  10  train_elbo=   28.9325  val_elbo=    6.1590  (recon_nll=   -1.8707, kl=  8.0298, global_logvar=-2.2517)
  Epoch  20  train_elbo=   42.7562  val_elbo=    1.9362  (recon_nll=   -7.3234, kl=  9.2596, global_logvar=-2.5471)
  Epoch  30  train_elbo=   21.0713  val_elbo=   -2.4630  (recon_nll=  -12.2615, kl=  9.7985, global_logvar=-2.6893)
  Epoch  40  train_elbo=   25.2666  val_elbo=   -8.3605  (recon_nll=  -19.1578, kl= 10.7972, global_logvar=-2.8711)
  Epoch  50  train_elbo=   30.1307  val_elbo=   -8.9899  (recon_nll=  -20.5388, kl= 11.5489, global_logvar=-3.0036)
  Epoch  60  train_elbo=   32.2717  val_elbo=    4.2496  (recon_nll=   -8.6937, kl= 12.9433, global_logvar=-2.8983)
  Early stopping at epoch 61 (no improvement for 10 epochs)

Finished at epoch 61, best val_elbo=-13.7691
Max train_elbo across all epochs: 69.92 (sho

In [26]:
print("=== VAE — scenario_A NLL by split/class ===")
vae_cic_A_scores = {
    "id":  score_split_vae(vae_cic_A, cic_A["id"],  CIC_FEATURES, DEVICE, "ID"),
    "id2": score_split_vae(vae_cic_A, cic_A["id2"], CIC_FEATURES, DEVICE, "ID2"),
    "ood": score_split_vae(vae_cic_A, cic_A["ood"], CIC_FEATURES, DEVICE, "OOD-attack"),
}
for name, df_scores in vae_cic_A_scores.items():
    print(f"{name}: inf={np.isinf(df_scores['vae_nll']).sum()}, nan={np.isnan(df_scores['vae_nll']).sum()}")

# Per-category breakdown, same as AE-A
ood_A_vae_scores = compute_vae_nll(vae_cic_A, cic_A["ood"], CIC_FEATURES, DEVICE)
ood_A_with_vae = cic_A["ood"][cic_A["ood"]["Class"] == 1].copy()
ood_A_with_vae["vae_nll"] = ood_A_vae_scores[cic_A["ood"]["Class"].values == 1]

print("\n=== VAE-A: Median NLL by unseen attack category (OOD-attack) ===")
print(ood_A_with_vae.groupby("Label")["vae_nll"].agg(["median", "count"]).sort_values("median"))

=== VAE — scenario_A NLL by split/class ===
ID: benign mean=-19.8670 median=-35.0855 | anomaly mean=1458.9711 median=701.1066 (n_benign=255042, n_anomaly=102542)
ID2: benign mean=-11.5709 median=-35.0816 | anomaly mean=1466.6913 median=670.2297 (n_benign=255042, n_anomaly=102543)
OOD-attack: benign mean=21.7820 median=-35.1389 | anomaly mean=1818.8031 median=10.6594 (n_benign=394777, n_anomaly=220656)
id: inf=0, nan=0
id2: inf=0, nan=0
ood: inf=0, nan=0

=== VAE-A: Median NLL by unseen attack category (OOD-attack) ===
              median   count
Label                       
PortScan  -33.057446   90694
Bot       -32.865223    1948
DDoS      523.274414  128014


In [27]:
AE_A_DIR = os.path.join(OUTPUT_DIR, "ae_cic_scenario_A")
os.makedirs(AE_A_DIR, exist_ok=True)
torch.save(ae_cic_A.state_dict(), os.path.join(AE_A_DIR, "model.pt"))
ae_cic_A_config = {
    "model": "Autoencoder", "dataset": "cic_ids2017", "scenario": "scenario_A (unseen-attack-type)",
    "architecture": ARCH_CONFIG_CIC, "training": TRAIN_CONFIG_CIC,
    "stopped_epoch": stopped_epoch_ae_cic_A, "best_val_loss": best_val_loss_ae_cic_A,
    "fit_size": len(fit_A), "val_size": len(val_A),
    "per_category_finding": {
        "description": "Bot and PortScan (unseen during training) remain near-indistinguishable from benign; DDoS (also unseen) is clearly detectable. Matches scenario_T's pattern for the same categories under temporal shift.",
        "median_by_category": {"Bot": 0.005198, "PortScan": 0.006745, "DDoS": 0.339020},
    },
}
with open(os.path.join(AE_A_DIR, "config.json"), "w") as f:
    json.dump(ae_cic_A_config, f, indent=2)
for split_name, scores_df in ae_cic_A_scores.items():
    scores_df.to_parquet(os.path.join(AE_A_DIR, f"scores_{split_name}.parquet"), index=False)
print("Saved:", AE_A_DIR, os.listdir(AE_A_DIR))

VAE_A_DIR = os.path.join(OUTPUT_DIR, "vae_cic_scenario_A")
os.makedirs(VAE_A_DIR, exist_ok=True)
torch.save(vae_cic_A.state_dict(), os.path.join(VAE_A_DIR, "model.pt"))
vae_cic_A_config = {
    "model": "VAE", "dataset": "cic_ids2017", "scenario": "scenario_A (unseen-attack-type)",
    "architecture": ARCH_CONFIG_CIC_VAE_A, "training": TRAIN_CONFIG_CIC,
    "stopped_epoch": stopped_epoch_vae_cic_A, "best_val_elbo": best_val_elbo_cic_A,
    "scoring_method": "negative log-likelihood, Monte Carlo averaged over 10 latent samples, global scalar output variance",
    "fit_size": len(fit_A), "val_size": len(val_A),
    "outlier_note": "Use median for ID/ID2/OOD comparisons, not mean (heavy-tailed, same as scenario_T).",
    "per_category_finding": {
        "description": "PortScan and Bot near-benign; DDoS clearly detectable. Consistent with AE-A and with scenario_T's VAE results across both models and both shift mechanisms.",
        "median_by_category": {"PortScan": -33.057446, "Bot": -32.865223, "DDoS": 523.274414},
    },
}
with open(os.path.join(VAE_A_DIR, "config.json"), "w") as f:
    json.dump(vae_cic_A_config, f, indent=2)
for split_name, scores_df in vae_cic_A_scores.items():
    scores_df.to_parquet(os.path.join(VAE_A_DIR, f"scores_{split_name}.parquet"), index=False)
print("Saved:", VAE_A_DIR, os.listdir(VAE_A_DIR))

Saved: /kaggle/working/models/ae_cic_scenario_A ['scores_id2.parquet', 'scores_ood.parquet', 'config.json', 'model.pt', 'scores_id.parquet']
Saved: /kaggle/working/models/vae_cic_scenario_A ['scores_id2.parquet', 'scores_ood.parquet', 'config.json', 'model.pt', 'scores_id.parquet']


In [28]:
import shutil
shutil.make_archive("/kaggle/working/models_archive_final", "zip", "/kaggle/working/models")
print("Size (MB):", os.path.getsize("/kaggle/working/models_archive_final.zip") / (1024*1024))

Size (MB): 23.220930099487305
